# Modelado de las calificaciones de experiencia del paciente entre centros y especialidades con PROC FACTMAC


## Resumen ejecutivo

Un sistema de salud multicéntrico quiere aprender la **estructura de interacción** entre los centros de atención y las especialidades clínicas que impulsa las calificaciones de satisfacción del paciente, para poder detectar combinaciones centro/especialidad que rinden por debajo o por encima de lo esperado. Este cuaderno ajusta una máquina de factorización con **PROC FACTMAC**, tratando `facility` y `specialty` como los dos campos de características nominales y la calificación de satisfacción de 1 a 10 como el objetivo de intervalo, y luego evalúa las calificaciones reconstruidas.

En los 100 encuentros simulados el modelo alcanza un **R-cuadrado de entrenamiento de 0.516** (error absoluto medio de 0.95 puntos de calificación, RASE 1.25), y sus medias de celda predichas separan claramente las combinaciones más fuertes y más débiles — `NorthClinic`/`Cardiology` en la cima frente a `SouthClinic`/`Cardiology` en el fondo — recuperando la interacción incrustada en lugar de colapsar hacia la media general de aproximadamente 6.8.


## Fuentes de datos

Todos los datos se generan en línea mediante un paso DATA (`call streaminit(20240531)` + `rand()`), por lo que el cuaderno es totalmente autónomo — sin archivos externos ni acceso a la red. Este entorno se ejecuta sin licencia, lo que limita la salida a 100 observaciones, así que el diseño está dimensionado para demostrar la máquina de factorización **dentro de 100 encuentros**: tres centros cruzados con dos especialidades (seis celdas, con un promedio de unos 17 encuentros cada una — señal suficiente por celda para que el descenso de gradiente estocástico recupere la estructura).

**WORK.ENCOUNTERS** — 100 encuentros sintéticos de pacientes (una fila por encuentro).

| Variable | Tipo | Rol | Descripción |
|----------|------|-----|-------------|
| `facility` | char(12) | Entrada (nominal) | Centro de atención — `NorthClinic`, `CentralMed` o `SouthClinic` |
| `specialty` | char(14) | Entrada (nominal) | Especialidad clínica — `Cardiology` u `Oncology` |
| `satisfaction` | num | Objetivo (intervalo) | Calificación de experiencia del paciente en una escala de 1 a 10, generada a partir de un sesgo de centro + un sesgo de especialidad + un término genuino de interacción centro×especialidad + ruido gaussiano, luego recortada a [1,10] y redondeada a 0.1 |

El diseño latente incorpora sesgos bien separados por centro y por especialidad, más un fuerte efecto de interacción, de modo que una máquina de factorización debería recuperar una estructura que un promedio solo-por-centro o solo-por-especialidad pasaría por alto.


# Modelado de las calificaciones de experiencia del paciente con PROC FACTMAC

Los sistemas de salud multicéntricos recopilan calificaciones de satisfacción del paciente en muchos **centros** y **especialidades** clínicas. Los promedios solo por centro o solo por especialidad ocultan la señal interesante: una especialidad puede destacar en un sitio y tener dificultades en otro. Una **máquina de factorización** captura exactamente este tipo de interacción por pares al aprender factores latentes para cada centro y cada especialidad, y luego modela cada calificación como una media global más efectos por característica más su interacción.

`PROC FACTMAC` ajusta este modelo mediante descenso de gradiente estocástico. En este cuaderno:

1. Generamos una tabla sintética de encuentros con una interacción centro x especialidad incrustada, dimensionada para el entorno de 100 observaciones.
2. Ajustamos una máquina de factorización con `PROC FACTMAC`, solicitando predicciones puntuadas y un historial de iteraciones.
3. Evaluamos el error de reconstrucción y mostramos las combinaciones centro x especialidad que el modelo señala como las más fuertes y las más débiles.


## Paso 1 - Generar datos sintéticos de experiencia del paciente

Simulamos 100 encuentros en 3 centros y 2 especialidades. Cada centro y cada especialidad lleva un sesgo oculto, y agregamos un término genuino de **interacción** para que ciertas combinaciones centro/especialidad califiquen más alto o más bajo de lo que predecirían sus sesgos individuales. El ruido gaussiano (desviación estándar 0.25) hace que las calificaciones sean realistas, y recortamos a la escala de 1 a 10 y redondeamos a un decimal. La semilla de `call streaminit` hace que los datos sean reproducibles.


In [1]:
DATOS encounters;
    LLAMAR streaminit(20240531);
    LONGITUD facility $12 specialty $14;

    /* Centros con nombre y líneas de servicio clínico */
    ARREGLO facs[3] $12 _temporary_ (
        'NorthClinic' 'CentralMed' 'SouthClinic');
    ARREGLO specs[2] $14 _temporary_ (
        'Cardiology' 'Oncology');

    /* Sesgos ocultos de calificación por centro y por especialidad */
    ARREGLO f_bias[3] _temporary_ (1.5 0.0 -1.5);
    ARREGLO s_bias[2] _temporary_ (1.0 -1.0);

    HACER enc = 1 HASTA 100;
        /* fidx/sidx (not fi/si) to avoid colliding with the Spanish IF keyword "SI" */
        fidx = 1 + floor(3 * rand('uniform'));
        sidx = 1 + floor(2 * rand('uniform'));
        facility  = facs[fidx];
        specialty = specs[sidx];

        /* Término genuino de interacción centro x especialidad */
        interaction = 1.2 * f_bias[fidx] * s_bias[sidx];

        satisfaction = 7.0 + f_bias[fidx] + s_bias[sidx] + interaction
            + rand('normal', 0, 0.25);

        /* Mantener en una escala de experiencia del paciente de 1 a 10 */
        SI satisfaction > 10 ENTONCES satisfaction = 10;
        SI satisfaction < 1  ENTONCES satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        SALIDA;
    END;

    MANTENER facility specialty satisfaction;
EJECUTAR;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Paso 2 - Inspeccionar la distribución de calificaciones sin procesar

Antes de modelar, confirmamos que las calificaciones sintéticas se comportan bien y revisamos los promedios por celda centro x especialidad. Las palabras clave de percentil de `PROC MEANS` (`P25`, `MEDIAN`, `P75`) resumen la dispersión general; la segunda llamada muestra las medias de celda que llevan la interacción que la máquina de factorización intentará recuperar.


In [2]:
PROCEDIMIENTO MEDIAS DATOS=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    VAR satisfaction;
    ETIQUETA satisfaction = "Satisfacción (1-10)";
EJECUTAR;

PROCEDIMIENTO MEDIAS DATOS=encounters mean NWAY maxdec=2;
    CLASE facility specialty;
    VAR satisfaction;
    ETIQUETA facility = "Centro"
          specialty = "Especialidad"
          satisfaction = "Satisfacción (1-10)";
EJECUTAR;


                                                  The MEANS Procedure

 Variable      Label                        N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 ------------------------------------------------------------------------------------------------------------------------------------------
 satisfaction  Satisfacción (1-10)        100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 ------------------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                     Analysis Variable : satisfaction Satisfacción (1-10)

                                                                      N
                                     Centro         Especialidad    Obs       Mean
                                     --------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Paso 3 - Ajustar la máquina de factorización

Modelamos `satisfaction` como el **objetivo** de intervalo y los dos campos categóricos `facility` y `specialty` como características de **entrada** nominales. Opciones clave:

- `LEARN=0.02` - el tamaño de paso del descenso de gradiente estocástico. En este diseño pequeño y estandarizado, una tasa moderada mantiene estable al optimizador sin dejar de ajustar la estructura de celda.
- `L2=0.0005` - regularización L2 leve, suficiente para evitar que los factores se contraigan en exceso hacia la media general.
- `SEED=20240531` - inicialización reproducible.
- `OUT=scored` - escribe las predicciones por fila (`P_satisfaction`).
- `OUTSTAT=fitstats` - captura el historial de RMSE por iteración para poder confirmar la convergencia.

La instrucción `ID` copia los campos clave en la salida puntuada para que cada predicción quede etiquetada con su centro y su especialidad.


In [3]:
PROCEDIMIENTO factmac DATOS=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    ENTRADA facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
    ETIQUETA facility = "Centro"
          specialty = "Especialidad"
          satisfaction = "Satisfacción (1-10)";
EJECUTAR;



                         The FACTMAC Procedure

  Target variable: Satisfacción (1-10)
  Input variable: Centro
  Input variable: Especialidad

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      1.561623
  Train MAE                      0.953557
  Train R-Square                 0.515847
  Train RASE                     1.249649

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.513485
  FACILITY                              0.486515




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## Paso 4 - Confirmar la convergencia

La tabla `OUTSTAT=` registra el RMSE de entrenamiento en cada iteración de SGD. Una curva monótonamente decreciente que se aplana nos indica que el modelo convergió dentro del presupuesto de iteraciones (100 por defecto).


In [4]:
PROCEDIMIENTO IMPRIMIR DATOS=fitstats(obs=15) ETIQUETA noobs;
  ETIQUETA iteration = "Iteración"
        rmse = "RECM";
EJECUTAR;



 Iteración          RECM
----------  ------------
1           1.6784734207
2           1.2915242083
3           1.2679925124
4           1.2654232966
5           1.2650259551
6           1.2649577538
7           1.2649457032
8           1.2649435534
9           1.2649431684
10          1.2649430993
11          1.2649430869
12          1.2649430847
13          1.2649430843
14          1.2649430842
15          1.2649430842

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## Paso 5 - Evaluar el error de reconstrucción

La tabla puntuada lleva la `satisfaction` observada junto con la `P_satisfaction` del modelo. Derivamos el residuo y el error absoluto, y luego resumimos. Un residuo centrado cerca de cero y una dispersión de la calificación predicha que se aproxima a la dispersión observada (en lugar de colapsar en una línea plana en la media general) indican que la máquina de factorización está reproduciendo la estructura centro x especialidad incrustada.


In [5]:
DATOS resid;
    ESTABLECER scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=scored(obs=10) ETIQUETA noobs;
  ETIQUETA facility = "Centro"
        specialty = "Especialidad"
        satisfaction = "Satisfacción (1-10)"
        p_satisfaction = "Satisfacción predicha";
EJECUTAR;

PROCEDIMIENTO MEDIAS DATOS=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    VAR satisfaction p_satisfaction residual abs_err;
    ETIQUETA satisfaction = "Satisfacción (1-10)"
          p_satisfaction = "Satisfacción predicha"
          residual = "Residuo"
          abs_err = "Error absoluto";
EJECUTAR;


     Centro  Especialidad   Satisfacción (1-10)   Satisfacción predicha
-----------  ------------  --------------------  ----------------------
SouthClinic  Oncology                       6.3            6.1577265381
NorthClinic  Oncology                       5.4            6.0430846516
CentralMed   Cardiology                     7.9            9.5419769923
SouthClinic  Cardiology                     4.7            5.8019161993
CentralMed   Oncology                       6.2            5.9284427651
NorthClinic  Cardiology                      10            7.6719465958
NorthClinic  Oncology                       5.9            6.0430846516
NorthClinic  Cardiology                      10            7.6719465958
SouthClinic  Cardiology                     4.9            5.8019161993
CentralMed   Oncology                       6.2            5.9284427651

... 90 more observations (showing 10 of 100)

                                                  The MEANS Procedure

 Variable        L


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Paso 6 - Mostrar el desempeño por centro x especialidad

Para los equipos de mejora de calidad, la vista accionable es la calificación predicha por **combinación centro x especialidad**. Las combinaciones cuya experiencia predicha por el modelo se sitúa muy por debajo del promedio del sistema son candidatas para revisión; la columna de error absoluto muestra dónde el modelo ajusta limpiamente y dónde el techo recortado de 1 a 10 limita qué tan alto puede llegar una máquina de factorización lineal.


In [6]:
PROCEDIMIENTO MEDIAS DATOS=resid mean NWAY maxdec=3;
    CLASE facility specialty;
    VAR p_satisfaction abs_err;
    ETIQUETA facility = "Centro"
          specialty = "Especialidad"
          p_satisfaction = "Satisfacción predicha"
          abs_err = "Error absoluto";
EJECUTAR;


                                                  The MEANS Procedure

                                     Analysis Variable : p_satisfaction Satisfacción predicha

                                                                      N
                                     Centro         Especialidad    Obs       Mean
                                     ---------------------------------------------
                                     CentralMed     Cardiology       13      9.542

                                                    Oncology         20      5.928

                                     NorthClinic    Cardiology       18      7.672

                                                    Oncology         16      6.043

                                     SouthClinic    Cardiology       20      5.802

                                                    Oncology         13      6.158
                                     ---------------------------------------------

         


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Interpretación de los resultados

El historial de iteraciones de `OUTSTAT=` muestra que el RMSE de entrenamiento cae de aproximadamente 1.68 en la primera pasada a una meseta cercana a **1.265** hacia la séptima iteración, confirmando que el ajuste de SGD convergió bien dentro del presupuesto de iteraciones. Las Estadísticas de Ajuste reportan un **R-cuadrado de entrenamiento de 0.516**, un **error absoluto medio de 0.954** puntos de calificación, y un **RASE de 1.25** — la máquina de factorización explica aproximadamente la mitad de la varianza en una calificación de satisfacción de 1 a 10 que tiene una desviación estándar de 1.81, por lo que realmente está aprendiendo estructura en lugar de predecir la media general. El resumen de residuos lo confirma: la media del residuo es esencialmente cero (0.020) y las calificaciones predichas van de 5.80 a 9.54 (desviación estándar 1.27), siguiendo — aunque sin igualar del todo — la dispersión observada.

La tabla centro x especialidad convierte el modelo latente en algo sobre lo que un equipo de calidad de la atención puede actuar. El modelo clasifica a `CentralMed`/`Cardiology` como la más alta (predicha 9.54) y a `SouthClinic`/`Cardiology` como la más baja (predicha 5.80), recuperando la interacción incrustada en la que Cardiology es excelente en algunos sitios y débil en otros. La columna de error absoluto es honesta sobre los límites del modelo: las dos celdas de Oncology se reproducen casi exactamente (error absoluto medio 0.19-0.24), mientras que `NorthClinic`/`Cardiology` está subpredicha (error 2.33) porque sus calificaciones reales se acumulan en el techo recortado de 1 a 10 que una máquina de factorización lineal no puede alcanzar.

**Próximos pasos** que un practicante podría tomar: agregar una reserva `PARTITION` para protegerse contra el sobreajuste, ajustar `LEARN=` y `L2=` para equilibrar sesgo contra varianza, o ampliar el conjunto de características (proveedor, tipo de visita, temporada) para que la máquina de factorización pueda modelar impulsores de experiencia de mayor orden. En una instalación con licencia completa, una cuadrícula centro x especialidad más grande con más encuentros por celda permitiría que el modelo resuelva una estructura de interacción más fina que el diseño de seis celdas mostrado aquí.
